# What is in a synthetic world

Three world families ship in `data/synthetic/worlds/`. They exist because
arche kept being blocked by data licensing: ai4privacy is academic-only and
forbids derivatives, i2b2 needs a data-use agreement, OpenSanctions is a
purchase. This is the path that needs nobody's permission.

| world | records | the question it exists to answer |
|---|---|---|
| `ng_supplier_v0` | 6,144 observations of 2,000 Nigerian organisations | does a matcher survive an entity *legitimately changing*: a supplier moves, a director resigns, a bank account is replaced |
| `artists_v0` | 9,944 observations of 2,905 real African musicians | what is a name-variant list worth, when `Wizkid` and `Ayodeji Ibrahim Balogun` are one person |
| `places_v1` | a 665-row master sheet and 1,000 delivery requests | does a delivery endpoint verify, ask one question, or refuse, and how often does it verify the wrong door |

The thing none of them share with a public benchmark: every disagreement
between two records carries **why it happened**, not just that it exists.

(Counts are of what a matcher is given. A manifest reports the entities the
generator *drew*, which is a larger number where some were never written to a
catalogue: 2,941 artists drawn, 2,905 with at least one observation.)

Run this from the repository root.

In [1]:
from arche_synthetic import load

w = load("data/synthetic/worlds/ng_supplier_v0")
w.describe()

ng_supplier_v0  seed 42  (records world)
  path        data\synthetic\worlds\ng_supplier_v0
  tables      differences 32,596, events 1,717, observations 6,144, truth 6,144
  matcher sees  observations.parquet; everything else is truth
  disagreements by kind  change 1,676, error 12,713, representation 18,207
  provenance  given and family names: synthetic-assumption


## What a matcher sees, and what it does not

One file is the input. Everything else is the answer key, and the generator
refuses to write a world where the two are mixed: `export.check_no_leak`
compares the observation columns against a forbidden list at write time, so a
truth column cannot reach the file a matcher is given even by accident.

`record_id` is a content hash for the same reason. If it encoded the entity id
the truth would be derivable from the input by string manipulation.

In [2]:
print("given to a matcher:", w.given_to_a_matcher)
print("tables:", ", ".join(f"{k} ({len(v):,})" for k, v in sorted(w.tables.items())))
print()
print("one observation:")
for key, value in list(w.observations[0].items())[:8]:
    print(f"  {key:<16} {value}")

given to a matcher: observations
tables: differences (32,596), events (1,717), observations (6,144), truth (6,144)

one observation:
  record_id        rec_d3a0207ed6524e5f
  source           erp
  observed_at      2025-10-10
  name             NAKAJJIGO TEXTILES VENTURES LIMITED
  rc_number        RC43614076
  tin              None
  phone            0806 428 6178
  email            None


## The column no other benchmark has

`differences.parquet` is one row per (pair, attribute): two records, the field
they disagree on, and the reason. The reasons fall into three kinds, and the
first one is the point.

| kind | example | are both records true? |
|---|---|---|
| `change` | the supplier moved; the director resigned | **yes**, at their respective times |
| `representation` | `Établissements Koné & Fils SARL` becomes `ETS Kone & Fils` | yes, the same fact encoded differently |
| `error` | a typo, an OCR misread, a truncation | no, one of them is a distortion |

Existing synthetic ER benchmarks produce only the third row: take a record,
corrupt it, call the result a duplicate. A matcher tuned on that learns that
all disagreement means "different entity", and then splits entities in
production the first time a supplier moves.

In [3]:
import pandas as pd

by_kind = w.difference_counts()
print({k: f"{v:,}" for k, v in sorted(by_kind.items())})

causes = pd.Series(w.difference_counts("cause")).sort_values(ascending=False)
causes.head(12).to_frame("rows")

{'change': '1,676', 'error': '12,713', 'representation': '18,207'}


,rows
case_upper,10069
ocr_confusion,4347
typo,4171
address_abbreviated,3415
truncation,3180
legal_suffix_abbreviated,3172
legal_suffix_dropped,1094
digit_mutation,1015
ORG_RELOCATED,917
ampersand,425


## One supplier that moved

The `change` rows are the ones worth looking at by hand. Take an entity whose
address changed, pull every observation of it in time order, and read what the
sources say. Both records are true; a matcher that reads the address and calls
them different entities has split a supplier in half.

In [4]:
moved = next(r for r in w.differences if r["cause"] == "ORG_RELOCATED")
records = w.records_of(moved["entity_id"])

print(f"entity {moved['entity_id']}, seen {len(records)} times\n")
for r in records:
    print(f"  {r['observed_at']}  {r['source']:<8} {r['name'][:38]:<38} {r['city']}")

print("\nwhy the first two disagree:")
for d in w.why(records[0]["record_id"], records[1]["record_id"]):
    print(f"  {d['attribute']:<14} {d['difference_kind']:<15} {d['cause']}")

entity syn:organisation:000002, seen 3 times

  2020-11-24  invoice  Slade Petroleum and Sons Ltd           Abuja
  2022-01-21  erp      Slade Petroleum an                     Port Harcourt
  2024-02-24  registry SLADE PETROLEUM AND SONS LIMITED       ABUJA

why the first two disagree:
  address        change          ORG_RELOCATED
  address        representation  address_abbreviated
  city           change          ORG_RELOCATED
  name           representation  legal_suffix_abbreviated
  name           error           truncation
  phone          error           typo


## The other two families

The same call opens them. `kind` says which family you have, because the
tables differ: an artist world has `collisions` (two different artists who
genuinely share a name) and no lifecycle events; a place world has a master
sheet and delivery requests rather than observations.

In [5]:
artists = load("data/synthetic/worlds/artists_v0")
artists.describe()
print()

places = load("data/synthetic/worlds/places_v1")
places.describe()

artists_v0  seed 42  (records world)
  path        data\synthetic\worlds\artists_v0
  tables      collisions 28, differences 7,656, observations 9,944, truth 9,944
  matcher sees  observations.parquet; everything else is truth
  disagreements by kind  error 818, representation 6,838
  provenance  artist alias groups (the truth): public-data-derived

places_v1  seed 42  (places world)
  path        data\synthetic\worlds\places_v1
  tables      requests 1,000, sheet 665, truth 2,000
  matcher sees  requests.parquet; everything else is truth
  renderings  ambiguous 195, confusable 91, confusable_typo 103, exact 595, landmark 241, landmark_only 177, partial 147, typo 261, unknown 123, unknown_confusable 67
  provenance  the master sheet: synthetic-assumption (which places exist); public-data-derived (the vocabulary they are made of)


The artist world's `collisions` table is what keeps the variant-list
experiment honest: a list that merges every name it knows about will merge
these too, and the cost has to be printed beside the gain.

In [6]:
pd.DataFrame(artists.collisions).head(5)

,entity_a,entity_b,shared_name,kind_a,kind_b
0,syn:artist:001018,syn:artist:001982,abdullah ibrahim,alias,filler
1,syn:artist:000158,syn:artist:000410,becky,alias,alias
2,syn:artist:000622,syn:artist:001964,david adeleke,alias,filler
3,syn:artist:000952,syn:artist:002126,desire sibanda,alias,filler
4,syn:artist:001250,syn:artist:001436,djoumessi tsague ulrich coco,alias,alias


## Reproducible, and saying so

Every world carries a manifest: the seed, the generator version, a content
fingerprint per table, and a provenance block per asset saying where the data
came from and what class of assumption it is. Two runs at the same seed agree
to the byte, which is checked by fingerprint rather than by comparing parquet
files (parquet embeds a writer version, so byte-identical data can produce
byte-different files).

In [7]:
m = w.manifest
print("seed:", m["seed"], " generator:", m["generator_version"])
print("fingerprints:")
for table, digest in m["content_fingerprints"].items():
    print(f"  {table:<14} {digest[:24]}...")
print("\nprovenance:")
for entry in m["provenance"]:
    print(f"  {entry['asset']}\n    {entry['class']}")

seed: 42  generator: 0.0.1
fingerprints:
  observations   c1aab11eec979854653f2481...
  truth          193c35f994dbb1f473a8f7d5...
  differences    1b9c716d3e7ba2bcea67c7fa...
  events         0ae1b0c65adce314989016af...

provenance:
  given and family names
    synthetic-assumption
  RC numbers and TINs
    public-data-derived


## Generate one yourself

Every world is a seed and a command. From `data/synthetic`:

```sh
python -m arche_synthetic --scale 400 --out worlds/small              # suppliers
python -m arche_synthetic --world-pack artists_v0 --scale 10000 --out worlds/artists_v0
python -m arche_synthetic --world-pack places_v1 --scale 1000 --out worlds/places_v1
```

`--scale` means organisations for the supplier world, records for artists and
requests for places. Each takes under ten seconds at full size.

## Where to read next

- **[27 What a variant list is worth](27_what_a_variant_list_is_worth.ipynb)**: the
  experiment Splink's author asked for, end to end.
- **[28 Does the ledger beat batch](28_does_the_ledger_beat_batch.ipynb)**: the seven
  arms over the supplier world, and the relocation finding.
- `data/synthetic/DATACARD.md` for every cause with its count, and what this
  data is not.